# A/B TESTING — KEBUTUHAN GIZI ANAK USIA 6–12 TAHUN Berdasarkan Standar Angka Kecukupan Gizi (AKG) Indonesia
# Proyek GiziKu — Coding Camp 2026 | CC26-PSU033
# Data Scientist: Cinta Wardana

KONTEKS BISNIS:

  Aplikasi nutrisi anak mengarahkan fitur utama ke segmen usia 6–12 tahun.
  A/B testing ini menguji apakah perbedaan kebutuhan gizi antar kelompok
  usia dan gender cukup signifikan untuk dijadikan dasar segmentasi fitur,
  personalisasi menu, dan threshold notifikasi yang berbeda.

STRUKTUR PENGUJIAN:

  Setiap hipotesis merepresentasikan satu keputusan desain produk nyata.
  Hasil statistik menentukan apakah segmentasi tersebut secara ilmiah
  dapat dipertanggungjawabkan atau tidak.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import ttest_ind, mannwhitneyu, chi2_contingency
import warnings
warnings.filterwarnings("ignore")

# 1. DATA AKG (Standar Kemenkes RI)

In [ ]:
df = pd.read_csv('akg_indonesia_final.csv')
df.head()

,Kategori,Label_Umur_Kondisi,Energi (kkal),Protein (g),Lemak (g),Lemak Omega 6 (g),Lemak Omega 3 (g),Karbohidrat (g),Serat (g),Natrium (mg),Air (ml),Is_Additional_Data
0,Bayi/Anak,0-5 bulan,550.0,9.0,31.0,4.4,0.5,59.0,0.0,120.0,700.0,False
1,Bayi/Anak,6-11 bulan,800.0,15.0,35.0,4.4,0.5,105.0,11.0,370.0,900.0,False
2,Bayi/Anak,1-3 tahun,1350.0,20.0,45.0,7.0,0.7,215.0,19.0,800.0,1150.0,False
3,Bayi/Anak,4-6 tahun,1400.0,25.0,50.0,10.0,0.9,220.0,20.0,900.0,1450.0,False
4,Bayi/Anak,7-9 tahun,1650.0,40.0,55.0,10.0,0.9,250.0,23.0,1000.0,1650.0,False


In [ ]:
AKG = {
    "4-6 tahun": {
        "energi": 1400, "protein": 25, "lemak": 50,
        "karbo": 220, "serat": 20
    },
    "7-9 tahun": {
        "energi": 1650, "protein": 40, "lemak": 55,
        "karbo": 250, "serat": 23
    },
    "10-12 tahun (L)": {
        "energi": 2000, "protein": 50, "lemak": 65,
        "karbo": 300, "serat": 28
    },
    "10-12 tahun (P)": {
        "energi": 1900, "protein": 55, "lemak": 65,
        "karbo": 280, "serat": 27
    },
}

# 2. SIMULASI POPULASI PENGGUNA

Kita simulasikan n=200 anak per kelompok dengan variasi asupan harian (noise ±15%) untuk mencerminkan variabilitas dunia nyata.
Seed tetap agar hasil reproducible.

In [ ]:
np.random.seed(42)
N_PER_GROUP = 200
NOISE_FACTOR = 0.15  # ±15% variasi asupan harian

def simulate_intake(akg_values: dict, n: int = N_PER_GROUP) -> pd.DataFrame:
    """
    Simulasikan asupan harian anak berdasarkan standar AKG dengan
    variabilitas Gaussian. Mencerminkan distribusi asupan nyata di populasi.
    """
    rows = []
    for group, nutrients in akg_values.items():
        for _ in range(n):
            row = {"kelompok": group}
            for nutrient, base in nutrients.items():
                noise = np.random.normal(1.0, NOISE_FACTOR)
                row[nutrient] = round(base * max(noise, 0.4), 1)
            rows.append(row)
    return pd.DataFrame(rows)

df = simulate_intake(AKG)

# 3. HELPER FUNCTIONS

In [ ]:
ALPHA = 0.05  # tingkat signifikansi 95%

def separator(char="─", width=70):
    return char * width

def cohen_d(a: np.ndarray, b: np.ndarray) -> float:
    """Effect size Cohen's d"""
    pooled_std = np.sqrt((np.std(a, ddof=1)**2 + np.std(b, ddof=1)**2) / 2)
    return (np.mean(a) - np.mean(b)) / pooled_std if pooled_std != 0 else 0.0

def interpret_effect(d: float) -> str:
    d = abs(d)
    if d < 0.2:   return "sangat kecil"
    elif d < 0.5: return "kecil"
    elif d < 0.8: return "sedang"
    else:         return "besar"

def two_sample_ttest(
    group_a: np.ndarray,
    group_b: np.ndarray,
    alpha: float = ALPHA
) -> dict:
    """Welch's t-test (tidak asumsikan equal variance)"""
    t_stat, p_val = ttest_ind(group_a, group_b, equal_var=False)
    d = cohen_d(group_a, group_b)
    ci_diff = stats.t.interval(
        1 - alpha,
        df=len(group_a) + len(group_b) - 2,
        loc=np.mean(group_a) - np.mean(group_b),
        scale=np.sqrt(np.var(group_a, ddof=1)/len(group_a) +
                      np.var(group_b, ddof=1)/len(group_b))
    )
    return {
        "t_stat": t_stat,
        "p_value": p_val,
        "reject_h0": p_val < alpha,
        "cohen_d": d,
        "effect_size": interpret_effect(d),
        "ci_diff": ci_diff,
        "mean_a": np.mean(group_a),
        "mean_b": np.mean(group_b),
        "std_a": np.std(group_a, ddof=1),
        "std_b": np.std(group_b, ddof=1),
    }

def print_result(label: str, result: dict, unit: str = "kkal"):
    verdict = "✅ TOLAK H₀ (ada perbedaan signifikan)" if result["reject_h0"] \
              else "❌ GAGAL TOLAK H₀ (tidak ada perbedaan signifikan)"
    pval_fmt = f"{result['p_value']:.2e}" if result["p_value"] < 0.001 \
               else f"{result['p_value']:.4f}"
    print(f"  Nutrien       : {label}")
    print(f"  Mean A        : {result['mean_a']:.1f} {unit}  (SD={result['std_a']:.1f})")
    print(f"  Mean B        : {result['mean_b']:.1f} {unit}  (SD={result['std_b']:.1f})")
    print(f"  Selisih mean  : {result['mean_a'] - result['mean_b']:+.1f} {unit}")
    print(f"  95% CI selisih: [{result['ci_diff'][0]:.1f}, {result['ci_diff'][1]:.1f}]")
    print(f"  t-statistic   : {result['t_stat']:.4f}")
    print(f"  p-value       : {pval_fmt}  (α={ALPHA})")
    print(f"  Cohen's d     : {result['cohen_d']:.4f}  [{result['effect_size']}]")
    print(f"  Keputusan     : {verdict}")

def implikasi(text: str):
    for line in text.strip().split("\n"):
        print(f"  💡 {line.strip()}")

# EKSEKUSI A/B TESTING

In [ ]:
print(separator("═"))
print("  A/B TESTING — GIZI ANAK 6–12 TAHUN (AKG Indonesia)")
print(f"  N per kelompok: {N_PER_GROUP}  |  α = {ALPHA}  |  Uji: Welch's t-test")
print(separator("═"))

results_summary = []  # untuk tabel ringkasan akhir

══════════════════════════════════════════════════════════════════════
  A/B TESTING — GIZI ANAK 6–12 TAHUN (AKG Indonesia)
  N per kelompok: 200  |  α = 0.05  |  Uji: Welch's t-test
══════════════════════════════════════════════════════════════════════


# HIPOTESIS 1
Apakah kebutuhan ENERGI anak 4–6 tahun berbeda secara signifikan dari anak 7–9 tahun?

# RELEVANSI PRODUK:
Menentukan apakah threshold kalori harian di notifikasi aplikasi perlu dibedakan antara kelompok 4–6 th vs 7–9 th, atau boleh menggunakan satu angka seragam.

In [ ]:
print(f"\n{separator()}")
print("  HIPOTESIS 1 — Energi: Anak 4–6 th vs 7–9 th")
print(separator())
print("  H₀ : μ_energi(4-6th) = μ_energi(7-9th)")
print("  H₁ : μ_energi(4-6th) ≠ μ_energi(7-9th)  [two-tailed]")
print()

a1 = df[df.kelompok == "4-6 tahun"]["energi"].values
b1 = df[df.kelompok == "7-9 tahun"]["energi"].values
r1 = two_sample_ttest(a1, b1)
print_result("Energi", r1, "kkal")

print()
print("  IMPLIKASI PRODUK:")
if r1["reject_h0"]:
    implikasi("""
    Notifikasi kalori HARUS dibedakan per kelompok usia.
    Gunakan threshold 1.400 kkal untuk anak 4–6 th, 1.650 kkal untuk 7–9 th.
    Fitur goal tracker perlu input usia wajib saat onboarding.
    """)
else:
    implikasi("Boleh gunakan threshold energi tunggal untuk kedua kelompok ini.")

results_summary.append({
    "No": 1, "Hipotesis": "Energi 4–6 th vs 7–9 th",
    "p-value": r1["p_value"], "Keputusan": "Tolak H₀" if r1["reject_h0"] else "Gagal Tolak",
    "Effect Size": r1["effect_size"]
})


──────────────────────────────────────────────────────────────────────
  HIPOTESIS 1 — Energi: Anak 4–6 th vs 7–9 th
──────────────────────────────────────────────────────────────────────
  H₀ : μ_energi(4-6th) = μ_energi(7-9th)
  H₁ : μ_energi(4-6th) ≠ μ_energi(7-9th)  [two-tailed]

  Nutrien       : Energi
  Mean A        : 1397.0 kkal  (SD=196.2)
  Mean B        : 1649.8 kkal  (SD=243.0)
  Selisih mean  : -252.8 kkal
  95% CI selisih: [-296.3, -209.4]
  t-statistic   : -11.4476
  p-value       : 2.79e-26  (α=0.05)
  Cohen's d     : -1.1448  [besar]
  Keputusan     : ✅ TOLAK H₀ (ada perbedaan signifikan)

  IMPLIKASI PRODUK:
  💡 Notifikasi kalori HARUS dibedakan per kelompok usia.
  💡 Gunakan threshold 1.400 kkal untuk anak 4–6 th, 1.650 kkal untuk 7–9 th.
  💡 Fitur goal tracker perlu input usia wajib saat onboarding.


# HIPOTESIS 2

Apakah kebutuhan PROTEIN anak 7–9 tahun berbeda signifikan dari anak 10–12 tahun (laki-laki)?

# RELEVANSI PRODUK:
Protein adalah nutrien terpenting untuk pertumbuhan. Ini menentukan apakah rekomendasi lauk-pauk di fitur meal planning perlu dipisah antara usia sekolah dasar awal dan atas.

In [ ]:
print(f"\n{separator()}")
print("  HIPOTESIS 2 — Protein: Anak 7–9 th vs 10–12 th (Laki-laki)")
print(separator())
print("  H₀ : μ_protein(7-9th) = μ_protein(10-12th L)")
print("  H₁ : μ_protein(7-9th) < μ_protein(10-12th L)  [one-tailed, directional]")
print()

a2 = df[df.kelompok == "7-9 tahun"]["protein"].values
b2 = df[df.kelompok == "10-12 tahun (L)"]["protein"].values
r2_full = two_sample_ttest(a2, b2)
# one-tailed: p/2 karena arah hipotesis sudah ditentukan
r2 = {**r2_full, "p_value": r2_full["p_value"] / 2,
      "reject_h0": (r2_full["p_value"] / 2) < ALPHA and r2_full["t_stat"] < 0}
print_result("Protein", r2, "g")

print()
print("  IMPLIKASI PRODUK:")
if r2["reject_h0"]:
    implikasi("""
    Rekomendasi menu tinggi protein berbeda nyata antar kelompok ini.
    Anak 10–12 th (L) butuh 50g vs hanya 40g pada anak 7–9 th (+25%).
    Segmentasi meal plan wajib: menu SD kelas 1–3 vs kelas 4–6.
    Badge 'Protein Cukup' harus gunakan threshold berbeda per usia.
    """)
else:
    implikasi("Meal plan protein dapat disamakan untuk dua kelompok ini.")

results_summary.append({
    "No": 2, "Hipotesis": "Protein 7–9 th vs 10–12 th (L)",
    "p-value": r2["p_value"], "Keputusan": "Tolak H₀" if r2["reject_h0"] else "Gagal Tolak",
    "Effect Size": r2["effect_size"]
})


──────────────────────────────────────────────────────────────────────
  HIPOTESIS 2 — Protein: Anak 7–9 th vs 10–12 th (Laki-laki)
──────────────────────────────────────────────────────────────────────
  H₀ : μ_protein(7-9th) = μ_protein(10-12th L)
  H₁ : μ_protein(7-9th) < μ_protein(10-12th L)  [one-tailed, directional]

  Nutrien       : Protein
  Mean A        : 40.1 g  (SD=6.8)
  Mean B        : 50.1 g  (SD=7.4)
  Selisih mean  : -10.1 g
  95% CI selisih: [-11.5, -8.7]
  t-statistic   : -14.1630
  p-value       : 1.98e-37  (α=0.05)
  Cohen's d     : -1.4163  [besar]
  Keputusan     : ✅ TOLAK H₀ (ada perbedaan signifikan)

  IMPLIKASI PRODUK:
  💡 Rekomendasi menu tinggi protein berbeda nyata antar kelompok ini.
  💡 Anak 10–12 th (L) butuh 50g vs hanya 40g pada anak 7–9 th (+25%).
  💡 Segmentasi meal plan wajib: menu SD kelas 1–3 vs kelas 4–6.
  💡 Badge 'Protein Cukup' harus gunakan threshold berbeda per usia.


# HIPOTESIS 3
Apakah ada perbedaan signifikan kebutuhan ENERGI antara laki-laki dan perempuan di usia 10–12 tahun?

# RELEVANSI PRODUK:
Menentukan apakah aplikasi perlu meminta input gender saat registrasi, atau cukup hanya usia. Ini berdampak pada kompleksitas onboarding dan akurasi personalisasi.

In [ ]:
print(f"\n{separator()}")
print("  HIPOTESIS 3 — Energi: 10–12 th Laki-laki vs Perempuan")
print(separator())
print("  H₀ : μ_energi(10-12 L) = μ_energi(10-12 P)")
print("  H₁ : μ_energi(10-12 L) ≠ μ_energi(10-12 P)  [two-tailed]")
print()

a3 = df[df.kelompok == "10-12 tahun (L)"]["energi"].values
b3 = df[df.kelompok == "10-12 tahun (P)"]["energi"].values
r3 = two_sample_ttest(a3, b3)
print_result("Energi", r3, "kkal")

print()
print("  IMPLIKASI PRODUK:")
if r3["reject_h0"]:
    implikasi("""
    Input GENDER wajib ditambahkan di onboarding untuk usia 10–12 th.
    Laki-laki 10–12 th butuh 2.000 kkal vs 1.900 kkal perempuan (+5.3%).
    A/B test onboarding: 'Step isi gender' vs 'skip' → ukur akurasi rekomendasi.
    Jika gender dilewati, gunakan nilai tengah 1.950 kkal sebagai default.
    """)
else:
    implikasi("Input gender tidak krusial untuk usia 10–12 th dari sisi kalori.")

results_summary.append({
    "No": 3, "Hipotesis": "Energi 10–12 th L vs P",
    "p-value": r3["p_value"], "Keputusan": "Tolak H₀" if r3["reject_h0"] else "Gagal Tolak",
    "Effect Size": r3["effect_size"]
})


──────────────────────────────────────────────────────────────────────
  HIPOTESIS 3 — Energi: 10–12 th Laki-laki vs Perempuan
──────────────────────────────────────────────────────────────────────
  H₀ : μ_energi(10-12 L) = μ_energi(10-12 P)
  H₁ : μ_energi(10-12 L) ≠ μ_energi(10-12 P)  [two-tailed]

  Nutrien       : Energi
  Mean A        : 2037.1 kkal  (SD=311.6)
  Mean B        : 1891.6 kkal  (SD=286.6)
  Selisih mean  : +145.5 kkal
  95% CI selisih: [86.6, 204.4]
  t-statistic   : 4.8600
  p-value       : 1.70e-06  (α=0.05)
  Cohen's d     : 0.4860  [kecil]
  Keputusan     : ✅ TOLAK H₀ (ada perbedaan signifikan)

  IMPLIKASI PRODUK:
  💡 Input GENDER wajib ditambahkan di onboarding untuk usia 10–12 th.
  💡 Laki-laki 10–12 th butuh 2.000 kkal vs 1.900 kkal perempuan (+5.3%).
  💡 A/B test onboarding: 'Step isi gender' vs 'skip' → ukur akurasi rekomendasi.
  💡 Jika gender dilewati, gunakan nilai tengah 1.950 kkal sebagai default.


# HIPOTESIS 4
Apakah ada perbedaan signifikan kebutuhan PROTEIN antara laki-laki dan perempuan di usia 10–12 tahun?

# RELEVANSI PRODUK:
Perempuan 10–12 th secara AKG butuh lebih banyak protein dari laki-laki (55g vs 50g). Ini counter-intuitive dan penting untuk divalidasi agar rekomendasi produk tidak salah arah.

In [ ]:
print(f"\n{separator()}")
print("  HIPOTESIS 4 — Protein: 10–12 th Perempuan vs Laki-laki")
print(separator())
print("  H₀ : μ_protein(10-12 P) = μ_protein(10-12 L)")
print("  H₁ : μ_protein(10-12 P) > μ_protein(10-12 L)  [one-tailed, P > L]")
print()

a4 = df[df.kelompok == "10-12 tahun (P)"]["protein"].values
b4 = df[df.kelompok == "10-12 tahun (L)"]["protein"].values
r4_full = two_sample_ttest(a4, b4)
r4 = {**r4_full, "p_value": r4_full["p_value"] / 2,
      "reject_h0": (r4_full["p_value"] / 2) < ALPHA and r4_full["t_stat"] > 0}
print_result("Protein", r4, "g")

print()
print("  IMPLIKASI PRODUK:")
if r4["reject_h0"]:
    implikasi("""
    TEMUAN PENTING: Perempuan 10–12 th butuh protein LEBIH TINGGI dari laki-laki.
    Ini karena percepatan pertumbuhan pubertas perempuan dimulai lebih awal.
    Rekomendasi lauk tinggi protein untuk perempuan 10–12 th harus lebih agresif.
    Konten edukasi di aplikasi: tambahkan insight 'fakta gizi remaja perempuan'.
    """)
else:
    implikasi("Rekomendasi protein dapat disamakan untuk L dan P di usia ini.")

results_summary.append({
    "No": 4, "Hipotesis": "Protein 10–12 th P > L",
    "p-value": r4["p_value"], "Keputusan": "Tolak H₀" if r4["reject_h0"] else "Gagal Tolak",
    "Effect Size": r4["effect_size"]
})


──────────────────────────────────────────────────────────────────────
  HIPOTESIS 4 — Protein: 10–12 th Perempuan vs Laki-laki
──────────────────────────────────────────────────────────────────────
  H₀ : μ_protein(10-12 P) = μ_protein(10-12 L)
  H₁ : μ_protein(10-12 P) > μ_protein(10-12 L)  [one-tailed, P > L]

  Nutrien       : Protein
  Mean A        : 54.8 g  (SD=8.5)
  Mean B        : 50.1 g  (SD=7.4)
  Selisih mean  : +4.6 g
  95% CI selisih: [3.1, 6.2]
  t-statistic   : 5.8165
  p-value       : 6.27e-09  (α=0.05)
  Cohen's d     : 0.5816  [sedang]
  Keputusan     : ✅ TOLAK H₀ (ada perbedaan signifikan)

  IMPLIKASI PRODUK:
  💡 TEMUAN PENTING: Perempuan 10–12 th butuh protein LEBIH TINGGI dari laki-laki.
  💡 Ini karena percepatan pertumbuhan pubertas perempuan dimulai lebih awal.
  💡 Rekomendasi lauk tinggi protein untuk perempuan 10–12 th harus lebih agresif.
  💡 Konten edukasi di aplikasi: tambahkan insight 'fakta gizi remaja perempuan'.


# HIPOTESIS 5
Apakah kebutuhan KARBOHIDRAT berbeda secara signifikan di seluruh
tiga kelompok usia (4–6, 7–9, 10–12 th)?
# RELEVANSI PRODUK:
Menggunakan one-way ANOVA untuk uji multi-grup sekaligus.
Menentukan apakah fitur "portion tracker" untuk nasi/roti perlu menampilkan target yang berbeda untuk setiap kelompok usia.

In [ ]:
print(f"\n{separator()}")
print("  HIPOTESIS 5 — Karbohidrat: ANOVA 3 Kelompok Usia")
print(separator())
print("  H₀ : μ_karbo(4-6) = μ_karbo(7-9) = μ_karbo(10-12)")
print("  H₁ : Minimal satu kelompok memiliki μ_karbo yang berbeda")
print()

k1 = df[df.kelompok == "4-6 tahun"]["karbo"].values
k2 = df[df.kelompok == "7-9 tahun"]["karbo"].values
k3 = df[df.kelompok == "10-12 tahun (L)"]["karbo"].values

f_stat, p_anova = stats.f_oneway(k1, k2, k3)
reject_anova = p_anova < ALPHA

print(f"  Nutrien       : Karbohidrat (g)")
print(f"  Mean 4–6 th   : {np.mean(k1):.1f} g  (SD={np.std(k1, ddof=1):.1f})")
print(f"  Mean 7–9 th   : {np.mean(k2):.1f} g  (SD={np.std(k2, ddof=1):.1f})")
print(f"  Mean 10–12 L  : {np.mean(k3):.1f} g  (SD={np.std(k3, ddof=1):.1f})")
print(f"  F-statistic   : {f_stat:.4f}")
print(f"  p-value       : {p_anova:.2e}  (α={ALPHA})")

if reject_anova:
    print(f"  Keputusan     : ✅ TOLAK H₀ — Ada perbedaan signifikan antar kelompok")
    print()
    print("  POST-HOC (Tukey HSD manual — perbandingan berpasangan):")
    pairs = [
        ("4–6 th", k1, "7–9 th", k2),
        ("4–6 th", k1, "10–12 L", k3),
        ("7–9 th", k2, "10–12 L", k3),
    ]
    for (la, ga, lb, gb) in pairs:
        t, p = ttest_ind(ga, gb, equal_var=False)
        sig = "✅ signifikan" if p < ALPHA else "❌ tidak signifikan"
        print(f"    {la} vs {lb}: Δ={np.mean(ga)-np.mean(gb):+.1f}g, p={p:.4f} → {sig}")
else:
    print(f"  Keputusan     : ❌ GAGAL TOLAK H₀")

print()
print("  IMPLIKASI PRODUK:")
if reject_anova:
    implikasi("""
    Portion tracker untuk nasi/roti/kentang WAJIB berbeda per kelompok usia.
    Target karbo: ~220g (4–6 th), ~250g (7–9 th), ~300g (10–12 th).
    Progress bar harian harus dikalibrasi berdasarkan target masing-masing kelompok.
    Ini juga mendukung fitur 'scan porsi makan' dengan referensi berbeda per usia.
    """)

results_summary.append({
    "No": 5, "Hipotesis": "Karbo ANOVA 3 kelompok",
    "p-value": p_anova, "Keputusan": "Tolak H₀" if reject_anova else "Gagal Tolak",
    "Effect Size": "besar" if reject_anova else "tidak ada"
})


──────────────────────────────────────────────────────────────────────
  HIPOTESIS 5 — Karbohidrat: ANOVA 3 Kelompok Usia
──────────────────────────────────────────────────────────────────────
  H₀ : μ_karbo(4-6) = μ_karbo(7-9) = μ_karbo(10-12)
  H₁ : Minimal satu kelompok memiliki μ_karbo yang berbeda

  Nutrien       : Karbohidrat (g)
  Mean 4–6 th   : 222.3 g  (SD=32.0)
  Mean 7–9 th   : 251.2 g  (SD=35.3)
  Mean 10–12 L  : 297.5 g  (SD=41.3)
  F-statistic   : 216.8752
  p-value       : 1.59e-71  (α=0.05)
  Keputusan     : ✅ TOLAK H₀ — Ada perbedaan signifikan antar kelompok

  POST-HOC (Tukey HSD manual — perbandingan berpasangan):
    4–6 th vs 7–9 th: Δ=-28.8g, p=0.0000 → ✅ signifikan
    4–6 th vs 10–12 L: Δ=-75.1g, p=0.0000 → ✅ signifikan
    7–9 th vs 10–12 L: Δ=-46.3g, p=0.0000 → ✅ signifikan

  IMPLIKASI PRODUK:
  💡 Portion tracker untuk nasi/roti/kentang WAJIB berbeda per kelompok usia.
  💡 Target karbo: ~220g (4–6 th), ~250g (7–9 th), ~300g (10–12 th).
  💡 Progress bar ha

# HIPOTESIS 6
Apakah proporsi anak yang memenuhi AKG energi berbeda antara kelompok 4–6 th dan 7–9 th, jika diasumsikan rata-rata asupan mereka adalah 90% dari kebutuhan?
# RELEVANSI PRODUK:
Menggunakan Chi-Square test untuk data kategoris (cukup / tidak cukup).
Mensimulasikan kondisi nyata di lapangan untuk melihat apakah prevalensi defisit gizi berbeda per kelompok — ini menentukan
prioritas intervensi notifikasi di aplikasi.

In [ ]:
print(f"\n{separator()}")
print("  HIPOTESIS 6 — Chi-Square: Proporsi Pemenuhan AKG Energi")
print(separator())
print("  Asumsi: Asupan rata-rata = 90% AKG (skenario defisit ringan)")
print("  H₀ : Proporsi pemenuhan AKG sama antara anak 4–6 th dan 7–9 th")
print("  H₁ : Proporsi pemenuhan AKG berbeda antar kedua kelompok")
print()

# Simulasikan dengan asupan aktual 90% dari AKG (tipikal populasi urban Indonesia)
np.random.seed(123)
INTAKE_RATIO = 0.90

def simulate_adequacy(akg_val, ratio, n, noise=0.18):
    intake = np.random.normal(akg_val * ratio, akg_val * noise, n)
    adequate = (intake >= akg_val).sum()
    return adequate, n - adequate

ade_46, inade_46 = simulate_adequacy(AKG["4-6 tahun"]["energi"], INTAKE_RATIO, N_PER_GROUP)
ade_79, inade_79 = simulate_adequacy(AKG["7-9 tahun"]["energi"], INTAKE_RATIO, N_PER_GROUP)

contingency = np.array([
    [ade_46, inade_46],
    [ade_79, inade_79]
])

chi2, p_chi2, dof, expected = chi2_contingency(contingency, correction=True)
reject_chi2 = p_chi2 < ALPHA

print(f"  Tabel Kontingensi:")
print(f"  {'Kelompok':<18} {'Cukup AKG':>12} {'Tidak Cukup':>12} {'Total':>8}")
print(f"  {'4–6 tahun':<18} {ade_46:>12} {inade_46:>12} {N_PER_GROUP:>8}")
print(f"  {'7–9 tahun':<18} {ade_79:>12} {inade_79:>12} {N_PER_GROUP:>8}")
print()
print(f"  % cukup 4–6 th : {ade_46/N_PER_GROUP*100:.1f}%")
print(f"  % cukup 7–9 th : {ade_79/N_PER_GROUP*100:.1f}%")
print(f"  Chi² statistic  : {chi2:.4f}")
print(f"  p-value         : {p_chi2:.4f}  (α={ALPHA})")
print(f"  Degrees of freedom: {dof}")

verdict_chi2 = "✅ TOLAK H₀ (proporsi berbeda signifikan)" if reject_chi2 \
               else "❌ GAGAL TOLAK H₀ (proporsi tidak berbeda signifikan)"
print(f"  Keputusan       : {verdict_chi2}")

print()
print("  IMPLIKASI PRODUK:")
if reject_chi2:
    implikasi("""
    Prevalensi defisit gizi berbeda nyata antar kelompok.
    Algoritma notifikasi harus memprioritaskan kelompok dengan defisit lebih tinggi.
    A/B test notifikasi: 'Reminder harian' vs 'Notifikasi cerdas berdasarkan pola makan'.
    """)
else:
    implikasi("""
    Strategi notifikasi tunggal dapat diterapkan untuk kedua kelompok usia ini.
    Fokus pada kualitas konten notifikasi, bukan frekuensi berbasis kelompok.
    """)

results_summary.append({
    "No": 6, "Hipotesis": "Proporsi pemenuhan AKG energi",
    "p-value": p_chi2, "Keputusan": "Tolak H₀" if reject_chi2 else "Gagal Tolak",
    "Effect Size": "kategoris (chi²)"
})


──────────────────────────────────────────────────────────────────────
  HIPOTESIS 6 — Chi-Square: Proporsi Pemenuhan AKG Energi
──────────────────────────────────────────────────────────────────────
  Asumsi: Asupan rata-rata = 90% AKG (skenario defisit ringan)
  H₀ : Proporsi pemenuhan AKG sama antara anak 4–6 th dan 7–9 th
  H₁ : Proporsi pemenuhan AKG berbeda antar kedua kelompok

  Tabel Kontingensi:
  Kelompok              Cukup AKG  Tidak Cukup    Total
  4–6 tahun                    57          143      200
  7–9 tahun                    51          149      200

  % cukup 4–6 th : 28.5%
  % cukup 7–9 th : 25.5%
  Chi² statistic  : 0.3171
  p-value         : 0.5734  (α=0.05)
  Degrees of freedom: 1
  Keputusan       : ❌ GAGAL TOLAK H₀ (proporsi tidak berbeda signifikan)

  IMPLIKASI PRODUK:
  💡 Strategi notifikasi tunggal dapat diterapkan untuk kedua kelompok usia ini.
  💡 Fokus pada kualitas konten notifikasi, bukan frekuensi berbasis kelompok.
